In [1]:
# This is based on Andrej Karpathys micrograd with extended classes, features, and functions

In [2]:
import math

In [3]:
# Value class for calculations + back prop
class Value:
    def __init__(self, data, _op="", _prev=()):
        self.data = data
        self.grad = 0.0
        self._op = _op
        self._prev = _prev
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, _op='+', _prev=(self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, _op ='*', _prev = (self, other))

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, _op='**', _prev=(self,))

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), _op='exp', _prev=(self,))

        def _backward():
            self.grad += math.exp(self.data) * out.grad

        out._backward = _backward
        return out
    
    def tanh(self):
        val = (math.exp(self.data) - math.exp(-self.data))/(math.exp(self.data) + math.exp(-self.data))
        out = Value(val, _op='tanh', _prev=(self,))

        def _backward():
            self.grad += (1 - val**2) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0, _op="relu", _prev=(self,))

        def _backward():
            self.grad += (1 if self.data > 0 else 0) * out.grad

        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return other + (-self)

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        return self * (other**-1)

    def backward(self):
        topo = []
        visited = set()
        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)
        build_topo(self)
        self.grad = 1.0
        
        for node in reversed(topo):
            node._backward()

        

In [4]:
import random

In [5]:
# Neuron class to initialize neurons with weights and biases
class Neuron:
    def __init__(self, nin):
        # Random gaussian(normal distribution) initialization for weights and bias
        self.weights = [Value(random.gauss(0,1)) for i in range(nin)]
        self.bias = Value(random.gauss(0,1))

    def __call__(self, x):
        #forward function, act = linear, then applies tanh
        act = sum((wi * xi for wi, xi in zip(self.weights, x)), self.bias)
        return act.tanh()

    def parameters(self):
        return self.weights + [self.bias]

In [6]:
# Creates a layer of neurons
class Layer:
    def __init__(self, nin, nout):
        # creates a layer/list of neurons
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        # runs forward pass for every neuron in layer
        out = [n(x) for n in self.neurons]
        # if only 1 neuron in layer, return the individual neuron, not list of a singular neuron
        return out[0] if len(out) == 1 else out

    def parameters(self):
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params

In [7]:
class MLP:
    def __init__(self, nin, nouts):
        self.layers = []
        for i, nout in enumerate(nouts):
            if i == 0:
                self.layers.append(Layer(nin, nout))
            else:
                self.layers.append(Layer(nouts[i-1], nout))
    def __call__(self, x):
        # runs forward pass for each layer in MLP
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for l in self.layers:
            params.extend(l.parameters())

        return params
        

In [8]:
# Training on simple dataset
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

# creates MLP with 2 layers, 4 neuron hidden, 1 neuron output
n = MLP(3, [4, 1])

epochs = 21
lr = 0.05
for k in range(epochs):
    # forward pass
    ypred = [n(x) for x in xs]
    # Calculating MSE
    loss = sum([(ygt - yout)**2 for yout, ygt in zip(ypred, ys)])

    for p in n.parameters():
        # zero gradients
        p.grad = 0.0
    loss.backward()

    for p in n.parameters():
        # back prop
        p.data -= lr * p.grad

    if k % 10 == 0:
        print(f"Epoch: {k} / {epochs} | loss: {loss.data}")

Epoch: 0 / 21 | loss: 14.580800015003078
Epoch: 10 / 21 | loss: 0.16389611559141717
Epoch: 20 / 21 | loss: 0.05456918545933145


In [9]:
print("\n--- Final Results ---")
for i, (yout, ygt) in enumerate(zip(ypred, ys)):
    print(f"Input {i+1} | Prediction: {yout.data:>7.4f} | Target: {ygt:>5.1f}")


--- Final Results ---
Input 1 | Prediction:  0.8644 | Target:   1.0
Input 2 | Prediction: -0.9194 | Target:  -1.0
Input 3 | Prediction: -0.8555 | Target:  -1.0
Input 4 | Prediction:  0.9061 | Target:   1.0
